# Merging ablation: possibilistic with and without model merging

Does merging similar models (Hellinger distance below a threshold) help the
possibilistic method? Each run below is the same pipeline with a different
`merge_threshold`; everything else is the default (no necessity). The
reference is **off** (`merge_threshold=None`, the default since 2026-11-19).

For every dataset and every threshold, `possibilistic_plain` with merging is
compared with merging off on the same days:

| Metric | Test | Better |
|---|---|---|
| Sharpe (unconstrained, long-only) | `sharpe_difference_test` (Anderson & Cheng GMM, Newey-West, 22 lags) | higher |
| Log-likelihood (daily) | `mean_difference_test` (Diebold–Mariano, Newey-West, 22 lags) | higher |
| Turnover (daily, unconstrained) | `mean_difference_test` | lower |

The runs (each folder's `config.json` records its settings and commit):

Calibration is shown descriptively (no test): mean Mahalanobis d² / n, which
is 1 for a calibrated Gaussian forecast; log-likelihood is the proper score
that tests it.

```
uv run python -m possibilistic_portfolio.pipeline_wrapper --out results_2026-11-19            # off (all algorithms)
# results_merge0.15: merging at 0.15 (all algorithms, made before the default changed)
uv run python -m possibilistic_portfolio.pipeline_wrapper --algorithms possibilistic --merge-threshold 0.10 --out results_merge0.10
uv run python -m possibilistic_portfolio.pipeline_wrapper --algorithms possibilistic --merge-threshold 0.224 --out results_merge0.224
# + the same with --simulated, then evaluation_pipeline <folder> for each
```

In [1]:
import json

import pandas as pd
from IPython.display import display

from possibilistic_portfolio.config import ROOT
from possibilistic_portfolio.portfolio_evaluation_functions import (
    mean_difference_test,
    portfolio_returns,
    sharpe_difference_test,
)

STEM = "possibilistic_plain"
N_LAGS = 22
ALPHA = 0.05
REFERENCE = "off"
RUNS = {  # label -> run folder; missing folders are skipped
    "off": ROOT / "results_2026-11-19",
    "0.10": ROOT / "results_merge0.10",
    "0.15": ROOT / "results_merge0.15",
    "0.224": ROOT / "results_merge0.224",
}

runs = {}
for label, folder in RUNS.items():
    cfg_path = folder / "config.json"
    if not cfg_path.exists():
        print(f"{label:>6}: {folder.name} missing, skipped")
        continue
    cfg = json.loads(cfg_path.read_text())
    codes = sorted({c["code"] for c in cfg["calls"]})
    print(
        f"{label:>6}: {folder.name}  merge_threshold={cfg['settings']['merge_threshold']}  code={codes}"
    )
    runs[label] = folder
assert REFERENCE in runs, "the merging-off run is needed"

   off: results_2026-11-19  merge_threshold=None  code=['d9d44db']
  0.10: results_merge0.10 missing, skipped
  0.15: results_merge0.15  merge_threshold=0.15  code=['599c77a', '92ba652 + uncommitted changes']
 0.224: results_merge0.224 missing, skipped


In [2]:
def dataset_folders(run):
    return {f.name: f for f in run.iterdir() if (f / "returns.csv").exists()}


def is_simulated(name):
    return not name[0].isdigit()  # French datasets start with 6_ / 10_


def load_series(folder, kind):
    # one strategy's daily series from an evaluated folder (log_likelihood.csv, ...)
    path = folder / f"{kind}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} missing: run evaluation_pipeline on {folder.parent.name}"
        )
    return pd.read_csv(path, index_col=0)[STEM]


def daily_scores(folder):
    returns = pd.read_csv(folder / "returns.csv", index_col=0)
    weights = {
        rule: pd.read_csv(folder / f"{STEM}{suffix}_weights.csv", index_col=0)
        for rule, suffix in [("unconstrained", ""), ("long-only", "_longonly")]
    }
    return {
        "returns": {
            rule: portfolio_returns(returns, w) for rule, w in weights.items()
        },
        "log_likelihood": load_series(folder, "log_likelihood"),
        "d2_over_n": float(
            (load_series(folder, "mahalanobis") ** 2).mean() / returns.shape[1]
        ),
        "turnover": weights["unconstrained"]
        .diff()
        .abs()
        .sum(axis=1, min_count=1),
    }

In [3]:
reference = dataset_folders(runs[REFERENCE])
rows, calibration = [], []
for label, run in runs.items():
    if label == REFERENCE:
        continue
    for name, folder in sorted(dataset_folders(run).items()):
        if name not in reference:
            continue
        a, b = daily_scores(folder), daily_scores(reference[name])
        base = dict(threshold=label, dataset=name, simulated=is_simulated(name))
        for rule in ("unconstrained", "long-only"):
            r = sharpe_difference_test(
                a["returns"][rule], b["returns"][rule], n_lags=N_LAGS
            )
            rows.append(
                base
                | dict(
                    metric=f"sharpe ({rule})",
                    merge=r.sharpe_strategy,
                    off=r.sharpe_benchmark,
                    t=r.t_statistic,
                    p=r.p_approx,
                )
            )
        calibration.append(
            base | dict(merge=a["d2_over_n"], off=b["d2_over_n"])
        )
        for metric in ("log_likelihood", "turnover"):
            r = mean_difference_test(a[metric], b[metric], n_lags=N_LAGS)
            rows.append(
                base
                | dict(
                    metric=metric,
                    merge=r.mean_a,
                    off=r.mean_b,
                    t=r.t_statistic,
                    p=r.p_approx,
                )
            )

results = pd.DataFrame(rows)
HIGHER_IS_BETTER = {"log_likelihood": True, "turnover": False}
results["merge_better"] = [
    (m - o > 0) == HIGHER_IS_BETTER.get(metric, True)
    for m, o, metric in zip(results["merge"], results["off"], results["metric"])
]
results["significant"] = results["p"] < ALPHA
print(f"{len(results)} comparisons, {results['dataset'].nunique()} datasets")

104 comparisons, 26 datasets


## Summary

Per threshold and metric: how many datasets favour merging or off, and how
many of those are significant at 5%. With ~26 datasets at 5%, about one
significant result per row is expected by chance; EW / VW versions of the
same portfolios overlap, so the datasets are not independent.

In [4]:
def summarise(df):
    g = df.groupby(["threshold", "metric"], sort=False)
    return pd.DataFrame(
        {
            "datasets": g.size(),
            "merge better": g["merge_better"].sum(),
            "sig. merge better": g.apply(
                lambda x: (x.significant & x.merge_better).sum()
            ),
            "sig. off better": g.apply(
                lambda x: (x.significant & ~x.merge_better).sum()
            ),
            "mean (merge - off)": g.apply(
                lambda x: (x["merge"] - x["off"]).mean()
            ),
        }
    )


for title, part in [
    ("Real data", results[~results.simulated]),
    ("Simulated", results[results.simulated]),
]:
    if len(part):
        print(title)
        display(summarise(part).round(4))

Real data


datasets  merge better  sig. merge better  \
threshold metric                                                              
0.15      sharpe (unconstrained)        20             8                  0   
          sharpe (long-only)            20             7                  0   
          log_likelihood                20             0                  0   
          turnover                      20            13                  7   

                                  sig. off better  mean (merge - off)  
threshold metric                                                       
0.15      sharpe (unconstrained)                6             -0.0033  
          sharpe (long-only)                    2             -0.0011  
          log_likelihood                       19             -0.0631  
          turnover                              2             -1.2622

Simulated


datasets  merge better  sig. merge better  \
threshold metric                                                              
0.15      sharpe (unconstrained)         6             2                  0   
          sharpe (long-only)             6             2                  0   
          log_likelihood                 6             4                  0   
          turnover                       6             2                  0   

                                  sig. off better  mean (merge - off)  
threshold metric                                                       
0.15      sharpe (unconstrained)                0             -0.0000  
          sharpe (long-only)                    0             -0.0000  
          log_likelihood                        0              0.0000  
          turnover                              2              0.0003

## Calibration: mean Mahalanobis d² / n

1 = calibrated; above 1 = the forecast covariance is too narrow (over-confident).

In [5]:
calib = pd.DataFrame(calibration)
display(
    calib.groupby(["threshold", "simulated"])[["merge", "off"]]
    .agg(["mean", "min", "max"])
    .round(3)
)

merge                  off              
                      mean    min    max   mean    min    max
threshold simulated                                          
0.15      False      0.978  0.835  1.331  1.005  0.842  1.339
          True       2.512  2.477  2.596  2.512  2.477  2.596

## Per dataset

In [6]:
with pd.option_context("display.max_rows", None):
    display(
        results.drop(columns=["merge_better"])
        .sort_values(["threshold", "metric", "dataset"])
        .round(4)
        .style.apply(
            lambda r: ["font-weight: bold" if r.significant else ""] * len(r),
            axis=1,
        )
    )

,threshold,dataset,simulated,metric,merge,off,t,p,significant
2,0.15,10_Industry_Portfolios_Daily_equal_weighted,False,log_likelihood,41.329300,41.388100,-5.039700,0.000000,True
6,0.15,10_Industry_Portfolios_Daily_value_weighted,False,log_likelihood,39.573300,39.620200,-3.438500,0.000600,True
10,0.15,10_Portfolios_Formed_on_ME_size_equal_weighted,False,log_likelihood,49.093600,49.156900,-3.648100,0.000300,True
14,0.15,10_Portfolios_Formed_on_ME_size_value_weighted,False,log_likelihood,50.355200,50.438700,-6.850900,0.000000,True
18,0.15,10_Portfolios_Formed_on_booktomarket_equal_weighted,False,log_likelihood,47.132200,47.180000,-4.401700,0.000000,True
22,0.15,10_Portfolios_Formed_on_booktomarket_value_weighted,False,log_likelihood,46.399600,46.461000,-3.484700,0.000500,True
26,0.15,10_Portfolios_Prior_momentum_equal_weighted,False,log_likelihood,47.903400,47.948000,-3.609600,0.000300,True
30,0.15,10_Portfolios_Prior_momentum_value_weighted,False,log_likelihood,44.339900,44.403100,-4.694000,0.000000,True
34,0.15,10_Portfolios_Prior_str_equal_weighted,False,log_likelihood,47.261400,47.294300,-1.805600,0.071000,False
38,0.15,10_Portfolios_Prior_str_value_weighted,False,log_likelihood,43.896500,43.936400,-4.080200,0.000000,True
